In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast
from pyspark.sql.types import IntegerType, DoubleType

# 1. Configuration de la session Spark pour la robustesse
spark = SparkSession.builder \
    .appName("Generation_Graphe_Riche_Transport") \
    .config("spark.sql.parquet.enableVectorizedReader", "false") \
    .config("spark.sql.parquet.mergeSchema", "true") \
    .config("spark.sql.shuffle.partitions", "300") \
    .getOrCreate()

In [ ]:
!pip install pyspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.3/455.3 MB 896.3 kB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.0/203.0 kB 21.5 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-4.1.0-py2.py3-none-any.whl size=455986285 sha256=debc4bfc718a49cda09e52653d845d58a0e12105f458c12a3f45a0bf50ef3c09
  Stored in directory: /root/.cache/pip/wheels/6b/9b/7c/2bea6ee44c4721d1af223365b8ab673a2db3ed4b759c12439d
Successfully built pyspark


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import gc
import glob
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# ===============================
# 1️⃣ CONFIGURATION
# ===============================
# Dossiers
INPUT_DIR = "/content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES_COSTS_1"
OUTPUT_DIR = "/content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES_COSTS_CLEAN"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Configuration Spark
spark = SparkSession.builder \
    .appName("Harmonisation_Structuree_GTFS") \
    .config("spark.sql.parquet.enableVectorizedReader", "false") \
    .config("spark.sql.parquet.mergeSchema", "false") \
    .getOrCreate()

# Colonnes
COLS_TO_DROP = ["psv", "busway", "motorcar", "highway", "junction", "oneway",
                "access", "smoothness", "maxspeed", "geometry_buffer", "geometry", "length", "lanes"]

COLS_TO_CAST_STRING = ["city", "from_stop_id", "to_stop_id", "id", "trip_id", "route_id",
                        "name", "route_long_name", "route_short_name", "width",
                        "from_arrival_time", "from_departure_time", "to_arrival_time", "to_departure_time"]

COLS_TO_CAST_DOUBLE = ["cost_highway", "cost_intersection", "cost_lanes", "cost_oneway",
                        "cost_motorcar", "cost_total", "cost_time_s", "travel_time_sec",
                        "travel_time_departure_sec", "delta_lat", "delta_lon", "distance_m_real",
                        "from_lat", "from_lon", "to_lat", "to_lon", "maxspeed_kmh",
                        "route_type", "index_right"]

# ===============================
# 2️⃣ FONCTION DE TRAITEMENT (Equiv. compute_multi_cost)
# ===============================
def process_city_file(df):
    # --- 1. Suppression des colonnes inutiles
    df = df.drop(*COLS_TO_DROP)

    # --- 2. Harmonisation String (Texte)
    for col_name in COLS_TO_CAST_STRING:
        if col_name in df.columns:
            df = df.withColumn(col_name, F.col(col_name).cast("string"))

    # --- 3. Harmonisation Double (Nombres)
    for col_name in COLS_TO_CAST_DOUBLE:
        if col_name in df.columns:
            df = df.withColumn(col_name, F.col(col_name).cast("double"))

    return df

# ===============================
# 3️⃣ LISTE DES VILLES ET GESTION DE LA REPRISE
# ===============================
# On liste les fichiers .parquet dans le dossier source
all_files_paths = glob.glob(f"{INPUT_DIR}/*.parquet")
# On extrait juste le nom de base (ex: 'Madrid_cost_chunk_0')
cities = [os.path.basename(f).replace(".parquet", "") for f in all_files_paths]

# On vérifie quels fichiers ont déjà le flag _DONE.txt dans le dossier destination
done_cities = {f.replace("_DONE.txt", "") for f in os.listdir(OUTPUT_DIR) if f.endswith("_DONE.txt")}

# Liste finale à traiter
cities_to_process = [c for c in cities if c not in done_cities]

print(f"✔ Total fichiers source : {len(cities)}")
print(f"✔ Fichiers déjà traités : {len(done_cities)}")
print(f"▶ Fichiers restants     : {len(cities_to_process)}")

# ===============================
# 4️⃣ BOUCLE DE TRAITEMENT AVEC REPRISE
# ===============================
for city_name in cities_to_process:
    print(f"\n🔹 Traitement Spark : {city_name}")

    input_path = os.path.join(INPUT_DIR, f"{city_name}.parquet")
    output_path = os.path.join(OUTPUT_DIR, f"{city_name}.parquet")
    done_flag = os.path.join(OUTPUT_DIR, f"{city_name}_DONE.txt")

    try:
        # 1. Lecture
        df = spark.read.parquet(input_path)

        # 2. Application des transformations
        df_final = process_city_file(df)

        # 3. Sauvegarde
        # On utilise coalesce(1) pour avoir 1 seul fichier de données par dossier
        df_final.coalesce(1).write.mode("overwrite").parquet(output_path)

        # 4. Création du Flag de réussite
        with open(done_flag, "w") as f:
            f.write("ok")

        print(f"✅ {city_name} sauvegardé et flaggé.")

        # --- NETTOYAGE MÉMOIRE ---
        del df, df_final
        gc.collect()
        spark.catalog.clearCache()

    except Exception as e:
        print(f"💥 Erreur sur {city_name} : {e}")
        gc.collect()

print(f"\n✨ TOUT EST TERMINÉ ! Les fichiers sont dans : {OUTPUT_DIR}")

✔ Total fichiers source : 539
✔ Fichiers déjà traités : 0
▶ Fichiers restants     : 539

🔹 Traitement Spark : Lurraldebus_-_Hernani_Urban_(bus_urbain_d’Hernani)_gtfs_osm_matched_cost_chunk_0
✅ Lurraldebus_-_Hernani_Urban_(bus_urbain_d’Hernani)_gtfs_osm_matched_cost_chunk_0 sauvegardé et flaggé.

🔹 Traitement Spark : Oñati_urbain_(Oñatiko_herribusa)_gtfs_osm_matched_cost_chunk_0
✅ Oñati_urbain_(Oñatiko_herribusa)_gtfs_osm_matched_cost_chunk_0 sauvegardé et flaggé.

🔹 Traitement Spark : Vectalia_Movilidad_(bus_de_la_ville_de_Cáceres)_gtfs_osm_matched_cost_chunk_0
✅ Vectalia_Movilidad_(bus_de_la_ville_de_Cáceres)_gtfs_osm_matched_cost_chunk_0 sauvegardé et flaggé.

🔹 Traitement Spark : Vectalia_Movilidad_(bus_de_la_ville_de_Cáceres)_gtfs_osm_matched_cost_chunk_1
✅ Vectalia_Movilidad_(bus_de_la_ville_de_Cáceres)_gtfs_osm_matched_cost_chunk_1 sauvegardé et flaggé.

🔹 Traitement Spark : Vectalia_Movilidad_(bus_de_la_ville_de_Cáceres)_gtfs_osm_matched_cost_chunk_2
✅ Vectalia_Movilida

In [ ]:
import os

# 📂 Dossier contenant les fichiers
CLEAN_DIR = "/content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES_COSTS_CLEAN"

# Récupérer tous les fichiers .parquet
files = [f for f in os.listdir(CLEAN_DIR) if f.endswith(".parquet")]

# Extraire le nom de la ville avant "_gtfs_osm_matched_cost_chunk"
cities = set()
for f in files:
    if "_gtfs_osm_matched_cost_chunk" in f:
        city = f.split("_gtfs_osm_matched_cost_chunk")[0]
        cities.add(city)

# Afficher les villes distinctes
print("✅ Villes distinctes trouvées :", len(cities))
for city in sorted(cities):
    print("-", city)


✅ Villes distinctes trouvées : 70
- AISA_(Bus_Madrid-Aranda_de_Duero-Burgo_de_Osma)
- AUCORSA_(Autobuses_de_Córdoba_S.A.)
- AUTNA_SL
- Alavabus
- Alvarez_Travelers_Coaches
- Ancebus
- Auif_Irunbus_(Lurraldebus)
- Autocorb_Coaches
- Autoridad_de_Transporte_Metropolitano_del_Area_de_Barcelona_(ATM)_Buses_and_trains_in_Catalonia_(full_version)
- Avanza_Grupo_(Mataró_city_bus)
- Bermibusa
- Bizkaibus
- BlaBlaCar_Bus
- Catalonia_Area_de_Barcelona
- Consorcio_Regional_de_Transportes_de_Madrid_CRTM_Madrid_City_Bus_(Autobus_urbano_de_Madrid)
- Cots_Alsina
- Direxis_Masats
- Direxis_TGO_(Transportes_Generales_de_Olesa)
- EMT_Tarragona
- El_Transbordador_de_Vizcaya
- El_Transbordador_de_Vizcaya_(Bizkaia_Bridge_Ferry)
- Empresa_Municipal_de_Transportes_de_Madrid_(EMT_Madrid)
- Empresa_Municipal_de_Transportes_de_Valencia_(EMT_Valencia)
- Etxebarri_Town_Hall
- FGV_-_Generalitat_Valenciana_Trains_and_trams_in_Alicante_and_the_Costa_Blanca
- Ferry_Fred._Olsen_(Fred_Olsen_Express)
- Funiculaire_de_

In [ ]:
!pip install pyspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.3/455.3 MB 959.7 kB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.0/203.0 kB 21.6 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-4.1.0-py2.py3-none-any.whl size=455986285 sha256=5c892046c474b248b5e868c4c19736fe8b908af847bad072b682791e93885941
  Stored in directory: /root/.cache/pip/wheels/6b/9b/7c/2bea6ee44c4721d1af223365b8ab673a2db3ed4b759c12439d
Successfully built pyspark


In [ ]:
import os
import gc
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# 1. Configuration de la session Spark
spark = SparkSession.builder \
    .appName("Graph_Generation_Safe_Loop") \
    .config("spark.sql.shuffle.partitions", "100") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

In [ ]:


spark.sparkContext.setCheckpointDir("/content/spark-checkpoints")

# --- CONFIGURATION DES CHEMINS ---
INPUT_PATH = "/content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES_COSTS_CLEAN/*.parquet"
OUTPUT_BASE = "/content/drive/MyDrive/GTFS_FINAL/GRAPH_FINAL_SECURE_V3"

os.makedirs(f"{OUTPUT_BASE}/NODES", exist_ok=True)
os.makedirs(f"{OUTPUT_BASE}/EDGES_TRAVEL", exist_ok=True)
os.makedirs(f"{OUTPUT_BASE}/EDGES_TRANSFER", exist_ok=True)

# --- FONCTIONS DE NETTOYAGE ---

def clean_city_name_logic(col):
    """Nettoie les noms de villes complexes (ex: Avanza_Mataro -> Mataro)"""
    c = F.regexp_replace(col, "_", " ")
    c = F.regexp_replace(c, "(?i)( city bus| urban| transport| trains and trams.*| intercity bus| coaches| buses.*| group.*| s.a.| sl)", "")
    return F.trim(c)

def time_to_seconds(col_name):
    parts = F.split(F.col(col_name), ":")
    return (parts[0].cast("int") * 3600 + parts[1].cast("int") * 60 + parts[2].cast("int"))

# --- 2. CHARGEMENT ET CALCUL DES POIDS RICHES ---

print("🚀 Préparation des données avec coût complexe...")
df = spark.read.parquet(INPUT_PATH)

# A. Nettoyage du nom de ville
df = df.withColumn("city_clean", clean_city_name_logic(F.col("city")))

# B. Calcul des temps
df = df.withColumn("dep_sec", time_to_seconds("from_departure_time")) \
       .withColumn("arr_sec", time_to_seconds("to_arrival_time"))

# C. Identification des Heures de Pointe
df = df.withColumn("is_peak",
    ((F.col("dep_sec") >= 25200) & (F.col("dep_sec") <= 32400)) |
    ((F.col("dep_sec") >= 61200) & (F.col("dep_sec") <= 68400))
)

# D. CONSERVATION DE VOTRE COÛT COMPLEXE + MULTIPLICATEUR
# On garde votre 'cost_total' qui contient déjà les junctions, lanes, highway, etc.
# On le multiplie par 2 en heure de pointe pour les bus
df = df.withColumn("weight_final",
    F.when(F.col("is_peak") & (F.col("route_type") == "3"), F.col("cost_total") * 2.0)
     .otherwise(F.col("cost_total"))
)

# --- 3. SAUVEGARDE DES NODES ET TRAVEL ---

# NODES
nodes = df.select(F.col("from_stop_id").alias("id"), "name",
                  F.col("city_clean").alias("city"),
                  F.col("from_lat").alias("lat"), F.col("from_lon").alias("lon")).distinct()
nodes.write.mode("overwrite").parquet(f"{OUTPUT_BASE}/NODES")

# TRAVEL EDGES (Segments de bus)
edges_travel = df.select(
    F.col("from_stop_id").alias("source"),
    F.col("to_stop_id").alias("target"),
    "trip_id", "route_id", "route_short_name", "route_type",
    F.col("city_clean").alias("city"),
    "dep_sec", "arr_sec", "weight_final", "cost_total"
).withColumn("edge_type", F.lit("TRAVEL"))

edges_travel.write.mode("overwrite").parquet(f"{OUTPUT_BASE}/EDGES_TRAVEL")

del df, nodes
gc.collect()



🚀 Préparation des données avec coût complexe...


342

In [ ]:
# --- 4. GÉNÉRATION DES TRANSFERTS AVEC PÉNALITÉ DE CONFORT ---
OUTPUT_BASE = "/content/drive/MyDrive/GTFS_FINAL/GRAPH_FINAL_SECURE_V3"
print("🔄 Génération des transferts intelligents...")
travel_data = spark.read.parquet(f"{OUTPUT_BASE}/EDGES_TRAVEL")
cities = [row['city'] for row in travel_data.select("city").distinct().collect() if row['city'] is not None]

for i, city in enumerate(cities):
    city_safe = city.replace(" ", "_").replace("/", "_")
    done_flag = f"{OUTPUT_BASE}/EDGES_TRANSFER/{city_safe}_DONE.txt"
    if os.path.exists(done_flag): continue

    try:
        city_df = travel_data.filter(F.col("city") == city).cache()

        arrivals = city_df.select(F.col("target").alias("stop_id"), F.col("trip_id").alias("tr_from"), F.col("arr_sec").alias("t_arr"))
        departures = city_df.select(F.col("source").alias("stop_id"), F.col("trip_id").alias("tr_to"), F.col("dep_sec").alias("t_dep"))

        # Jointure : on limite l'attente à 20 min max
        transfers = arrivals.join(departures, "stop_id") \
            .filter((F.col("tr_from") != F.col("tr_to")) &
                    (F.col("t_dep") >= F.col("t_arr") + 120) &
                    (F.col("t_dep") <= F.col("t_arr") + 1200))

        # LOGIQUE DE POIDS POUR LE TRANSFERT
        # Votre 'cost_total' pour un segment est environ entre 1 et 10.
        # Pour qu'un transfert soit "cher", on lui donne un poids de 15.0 (équivalent à un gros trajet)
        # Cela force Dijkstra à rester dans le bus direct.
        final_transfers = transfers.withColumn("weight_final", ((F.col("t_dep") - F.col("t_arr")) / 60) + 15.0) \
            .select(
                F.col("stop_id").alias("source"),
                F.col("stop_id").alias("target"),
                F.col("tr_from").alias("trip_id_from"),
                F.col("tr_to").alias("trip_id_to"),
                F.lit(city).alias("city"),
                F.col("t_arr").alias("dep_sec"),
                F.col("t_dep").alias("arr_sec"),
                "weight_final"
            ).withColumn("edge_type", F.lit("TRANSFER"))

        output_city_path = f"{OUTPUT_BASE}/EDGES_TRANSFER/city={city_safe}"
        final_transfers.coalesce(1).write.mode("overwrite").parquet(output_city_path)

        with open(done_flag, "w") as f: f.write("ok")
        city_df.unpersist()
        gc.collect()
        print(f"✅ {city} terminé")

    except Exception as e:
        print(f"❌ Erreur {city}: {e}")

print(f"✨ GRAPHE RICHE GÉNÉRÉ DANS : {OUTPUT_BASE}")

🔄 Génération des transferts intelligents...
✅ dBus (Donostiabus) terminé
✅ Junta de Extremadura (Bus du gouvernement régional d’Estrémadure) terminé
✅ Catalonia Area de Barcelona terminé
✅ Ouigo terminé
✅ Alavabus terminé
✅ Empresa Municipal dees de Madrid (EMT Madrid) terminé
✅ Autocorb terminé
✅ Generalitat of Catalonia (Intercity bus) terminé
✅ TUS (Transportesos de Santander) terminé
✅ Empresa Municipal dees de Valencia (EMT Valencia) terminé
✅ La Veloz SA (Buses from the Belchite countryside area to Zaragoza (C12)) terminé
✅ Direxis TGO (Transportes Generales de Olesa) terminé
✅ Autoridad dee Metropolitano del Area de Barcelona (ATM) terminé
✅ Lurraldebus Ekialdebus terminé
✅ La Coruña Tram Company SA terminé
✅ AISA (Bus Madrid-Aranda de Duero-Burgo de Osma) terminé
✅ Bizkaibus terminé
✅ Consorcio Regional dees de Madrid CRTM Madrid (Autobuso de Madrid) terminé
✅ Etxebarri Town Hall terminé
✅ Lurraldebus TBH (Interurban Tolosa Buruntzaldea) terminé
✅ Direxis Masats terminé
✅ Pa

In [ ]:
from pyspark.sql import functions as F

# --- CHEMINS ---
PATH_NODES = "/content/drive/MyDrive/GTFS_FINAL/GRAPH_FINAL_SECURE_V3/NODES"
PATH_TRAVEL = "/content/drive/MyDrive/GTFS_FINAL/GRAPH_FINAL_SECURE_V3/EDGES_TRAVEL"

# 1. Extraire les villes des NODES (Stations)
print("🏙️ Villes dans la table NODES :")
cities_nodes = spark.read.parquet(PATH_NODES) \
    .select("city") \
    .distinct() \
    .orderBy("city") \
    .collect()

for row in cities_nodes:
    print(f"- {row['city']}")

print("-" * 50)

# 2. Extraire les villes des EDGES_TRAVEL (Lignes de bus)
print("🚌 Villes dans la table EDGES_TRAVEL :")
cities_travel = spark.read.parquet(PATH_TRAVEL) \
    .select("city") \
    .distinct() \
    .orderBy("city") \
    .collect()

for row in cities_travel:
    print(f"- {row['city']}")

# 3. Vérification de la cohérence
set_nodes = set([r['city'] for r in cities_nodes])
set_travel = set([r['city'] for r in cities_travel])

diff = set_nodes.symmetric_difference(set_travel)
if not diff:
    print("\n✅ Parfait : Les noms de villes correspondent exactement entre les deux tables !")
else:
    print(f"\n⚠️ Attention : {len(diff)} noms ne correspondent pas entre les deux tables.")
    print(f"Différences : {diff}")

🏙️ Villes dans la table NODES :
- AISA (Bus Madrid-Aranda de Duero-Burgo de Osma)
- AUCORSA (Autobuses de Córdoba)
- AUTNA
- Alavabus
- Alvarez Travelers
- Ancebus
- Auif Irunbus (Lurraldebus)
- Autocorb
- Autoridad dee Metropolitano del Area de Barcelona (ATM)
- Avanza Grupo (Mataró)
- Bermibusa
- Bizkaibus
- BlaBlaCar Bus
- Catalonia Area de Barcelona
- Consorcio Regional dees de Madrid CRTM Madrid (Autobuso de Madrid)
- Cots Alsina
- Direxis Masats
- Direxis TGO (Transportes Generales de Olesa)
- EMT Tarragona
- El Transbordador de Vizcaya
- El Transbordador de Vizcaya (Bizkaia Bridge Ferry)
- Empresa Municipal dees de Madrid (EMT Madrid)
- Empresa Municipal dees de Valencia (EMT Valencia)
- Etxebarri Town Hall
- FGV - Generalitat Valenciana
- Ferry Fred. Olsen (Fred Olsen Express)
- Funiculaire de Artxanda
- Generalitat of Catalonia (Intercity bus)
- Gobierno de Cantabria (Bus en Cantabrie)
- Junta de Extremadura (Bus du gouvernement régional d’Estrémadure)
- Kbus (Barakaldo)
-

In [ ]:
import os

# 📂 Dossier des transferts
TRANSFER_DIR = "/content/drive/MyDrive/GTFS_FINAL/GRAPH_FINAL_SECURE_V3/EDGES_TRANSFER"

# Récupérer uniquement les fichiers qui finissent par _DONE.txt
done_files = [f for f in os.listdir(TRANSFER_DIR) if f.endswith("_DONE.txt")]

# Extraire le nom de la ville (avant _DONE.txt)
cities_done = [f.replace("_DONE.txt", "") for f in done_files]

print("✅ Nombre de villes traitées :", len(cities_done))
print("📍 Liste des villes :")
for city in sorted(cities_done):
    print("-", city)


✅ Nombre de villes traitées : 70
📍 Liste des villes :
- AISA_(Bus_Madrid-Aranda_de_Duero-Burgo_de_Osma)
- AUCORSA_(Autobuses_de_Córdoba)
- AUTNA
- Alavabus
- Alvarez_Travelers
- Ancebus
- Auif_Irunbus_(Lurraldebus)
- Autocorb
- Autoridad_dee_Metropolitano_del_Area_de_Barcelona_(ATM)
- Avanza_Grupo_(Mataró)
- Bermibusa
- Bizkaibus
- BlaBlaCar_Bus
- Catalonia_Area_de_Barcelona
- Consorcio_Regional_dees_de_Madrid_CRTM_Madrid_(Autobuso_de_Madrid)
- Cots_Alsina
- Direxis_Masats
- Direxis_TGO_(Transportes_Generales_de_Olesa)
- EMT_Tarragona
- El_Transbordador_de_Vizcaya
- El_Transbordador_de_Vizcaya_(Bizkaia_Bridge_Ferry)
- Empresa_Municipal_dees_de_Madrid_(EMT_Madrid)
- Empresa_Municipal_dees_de_Valencia_(EMT_Valencia)
- Etxebarri_Town_Hall
- FGV_-_Generalitat_Valenciana
- Ferry_Fred._Olsen_(Fred_Olsen_Express)
- Funiculaire_de_Artxanda
- Generalitat_of_Catalonia_(Intercity_bus)
- Gobierno_de_Cantabria_(Bus_en_Cantabrie)
- Junta_de_Extremadura_(Bus_du_gouvernement_régional_d’Estrémadure

In [ ]:
import duckdb
import networkx as nx
import pandas as pd
import time
import gc
from geopy.geocoders import Nominatim
from scipy.spatial import KDTree

# --- 1. CONFIGURATION ---
PATH_BASE = "/content/drive/MyDrive/GTFS_FINAL/GRAPH_FINAL_SECURE_V3"
VILLE_NAME = "Bizkaibus"
VILLE_FOLDER = VILLE_NAME.replace(" ", "_").replace("(", "").replace(")", "")

# On réduit la fenêtre à 1h (3600s) pour économiser la RAM
HEURE_DEPART = "8:00:00"
h, m, s = map(int, HEURE_DEPART.split(':'))
t_sec = h * 3600 + m * 60 + s

con = duckdb.connect()
# Augmentation du timeout pour éviter le "Read timed out"
geolocator = Nominatim(user_agent="final_checker_v2", timeout=20)

print(f"🚀 Démarrage du test pour : {VILLE_NAME}")

# --- 2. LECTURE DUCKDB (Filtrage strict pour la RAM) ---
print("⌛ Chargement des données...")

# Stations
df_nodes = con.execute(f"SELECT id, name, lat, lon FROM read_parquet('{PATH_BASE}/NODES/*.parquet') WHERE city = '{VILLE_NAME}'").df()

# Trajets : On limite strictement à 1h après le départ
df_travel = con.execute(f"""
    SELECT source, target, weight_final, route_type, dep_sec, arr_sec
    FROM read_parquet('{PATH_BASE}/EDGES_TRAVEL/*.parquet')
    WHERE city = '{VILLE_NAME}' AND dep_sec BETWEEN {t_sec} AND {t_sec + 3600*2}
""").df()

# Transferts : On ne charge que si nécessaire
try:
    df_transfer = con.execute(f"""
        SELECT source, target, weight_final, dep_sec, arr_sec
        FROM read_parquet('{PATH_BASE}/EDGES_TRANSFER/city={VILLE_FOLDER}/*.parquet')
        WHERE dep_sec BETWEEN {t_sec} AND {t_sec + 3600}
    """).df()
except:
    df_transfer = pd.DataFrame()

print(f"✅ Données en mémoire (Travel: {len(df_travel)} / Transfer: {len(df_transfer)})")

# --- 3. GÉOCODAGE SÉCURISÉ ---
spatial_tree = KDTree(df_nodes[['lat', 'lon']].values)

def get_node_safe(q):
    try:
        print(f"🌍 Recherche de l'adresse : {q}...")
        loc = geolocator.geocode(f"{q}, Bizkaia, Spain")
        if loc:
            idx = spatial_tree.query([loc.latitude, loc.longitude])[1]
            return df_nodes.iloc[idx]
        return None
    except Exception as e:
        print(f"⚠️ Erreur Geocoder pour {q}: {e}")
        return None

node_start = get_node_safe("Abando")
node_end = get_node_safe("Deusto")

if node_start is None or node_end is None:
    print("❌ Impossible de localiser les quartiers. Test avec IDs par défaut...")
    # IDs de secours si le géocodage échoue encore
    S_ID, T_ID = df_nodes['id'].iloc[0], df_nodes['id'].iloc[-1]
else:
    S_ID, T_ID = node_start['id'], node_end['id']

# --- 4. CONSTRUCTION DU GRAPHE (Optimisé itertuples) ---
G = nx.DiGraph()

for r in df_travel.itertuples():
    G.add_edge(r.source, r.target, weight=r.weight_final, mode=r.route_type, dep=r.dep_sec, arr=r.arr_sec)

for r in df_transfer.itertuples():
    G.add_edge(r.source, r.target, weight=r.weight_final, mode='TRANSFER', dep=r.dep_sec, arr=r.arr_sec)

# Libération immédiate de la RAM
del df_travel, df_transfer
gc.collect()

# --- 5. RECOMMANDATION ---
try:
    path = nx.shortest_path(G, S_ID, T_ID, weight='weight')

    print(f"\n✅ ITINÉRAIRE TROUVÉ")
    print(f"{'='*90}")
    print(f"{'STATION':<35} | {'MODE':<12} | {'HEURE'}")
    print(f"{'-'*90}")

    for i in range(len(path)-1):
        d = G.get_edge_data(path[i], path[i+1])
        s_name = df_nodes[df_nodes['id'] == path[i]]['name'].values[0]
        mode = "Bus" if d['mode'] == 3.0 else ("Tram" if d['mode'] == 0.0 else "Correspondance")
        print(f"📍 {s_name[:34]:<35} | {mode:<12} | {time.strftime('%H:%M:%S', time.gmtime(d['dep']))}")

    arr_f = G.get_edge_data(path[-2], path[-1])['arr']
    print(f"{'-'*90}")
    print(f"🏁 ARRIVÉE FINALE : {time.strftime('%H:%M:%S', time.gmtime(arr_f))}")
    print(f"⏱️ DURÉE : {(arr_f - t_sec)//60} min")
    print(f"{'='*90}")

except Exception as e:
    print(f"❌ Aucun chemin trouvé : {e}")

con.close()

🚀 Démarrage du test pour : Bizkaibus
⌛ Chargement des données...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Données en mémoire (Travel: 157866 / Transfer: 6240918)
🌍 Recherche de l'adresse : Abando...
🌍 Recherche de l'adresse : Deusto...
❌ Aucun chemin trouvé : Source 607 is not in G


In [ ]:
import duckdb
import networkx as nx
import pandas as pd
import time
import gc
from geopy.geocoders import Nominatim
from scipy.spatial import KDTree

# --- 1. CONFIGURATION ---
PATH_BASE = "/content/drive/MyDrive/GTFS_FINAL/GRAPH_FINAL_SECURE_V3"
VILLE_NAME = "Catalonia Area de Barcelona"
VILLE_FOLDER = VILLE_NAME.replace(" ", "_").replace("(", "").replace(")", "")

HEURE_DEPART = "08:00:00"
h, m, s = map(int, HEURE_DEPART.split(':'))
t_sec = h * 3600 + m * 60 + s

con = duckdb.connect()
geolocator = Nominatim(user_agent="bcn_final_test", timeout=20)

print(f"🚀 Recherche optimisée pour Barcelone à {HEURE_DEPART}")

# --- 2. LECTURE DUCKDB FILTRÉE (SÉCURITÉ RAM) ---
# On charge 2h de données mais on filtre très fort les transferts
df_nodes = con.execute(f"SELECT id, name, lat, lon FROM read_parquet('{PATH_BASE}/NODES/*.parquet') WHERE city = '{VILLE_NAME}'").df()

df_travel = con.execute(f"""
    SELECT source, target, weight_final, route_type, route_id, dep_sec, arr_sec
    FROM read_parquet('{PATH_BASE}/EDGES_TRAVEL/*.parquet')
    WHERE city = '{VILLE_NAME}' AND dep_sec BETWEEN {t_sec} AND {t_sec + 7200}
""").df()

# On limite les transferts à 15 min max pour réduire les 17 millions de lignes
df_transfer = con.execute(f"""
    SELECT source, target, weight_final, dep_sec, arr_sec
    FROM read_parquet('{PATH_BASE}/EDGES_TRANSFER/city={VILLE_FOLDER}/*.parquet')
    WHERE dep_sec BETWEEN {t_sec} AND {t_sec + 3600}
    AND (arr_sec - dep_sec) <= 900
""").df()

print(f"✅ Données chargées : {len(df_travel)} trajets | {len(df_transfer)} transferts légers")

# --- 3. RECHERCHE DES 5 STATIONS LES PLUS PROCHES ---
spatial_tree = KDTree(df_nodes[['lat', 'lon']].values)

def get_nearby_nodes(q, k=5):
    loc = geolocator.geocode(f"{q}, Barcelona, Spain")
    if not loc: return []
    dists, indices = spatial_tree.query([loc.latitude, loc.longitude], k=k)
    return df_nodes.iloc[indices]

print("🌍 Localisation des quartiers...")
nodes_start = get_nearby_nodes("Placa de Catalunya")
nodes_end = get_nearby_nodes("Sagrada Familia")

# --- 4. CONSTRUCTION DU GRAPHE ---
G = nx.DiGraph()
for r in df_travel.itertuples():
    G.add_edge(r.source, r.target, weight=r.weight_final, mode=r.route_type, line=r.route_id, dep=r.dep_sec, arr=r.arr_sec, type='TRAVEL')
for r in df_transfer.itertuples():
    G.add_edge(r.source, r.target, weight=r.weight_final, mode='TRANSFER', dep=r.dep_sec, arr=r.arr_sec, type='TRANSFER')

# --- 5. CALCUL DU MEILLEUR CHEMIN PARMI LES POSSIBILITÉS ---
print("⚙️ Analyse des combinaisons de stations...")
path = None
for s_node in nodes_start['id']:
    for t_node in nodes_end['id']:
        if s_node in G and t_node in G:
            try:
                path = nx.shortest_path(G, s_node, t_node, weight='weight')
                break
            except: continue
    if path: break

# --- 6. AFFICHAGE ---
if path:
    def fmt(s): return time.strftime('%H:%M:%S', time.gmtime(s))
    print(f"\n✅ ITINÉRAIRE TROUVÉ")
    print(f"{'='*100}")
    print(f"{'STATION':<35} | {'MODE':<12} | {'LIGNE':<10} | {'ARRIVÉE RÉELLE'}")
    print(f"{'-'*100}")
    for i in range(len(path)-1):
        d = G.get_edge_data(path[i], path[i+1])
        s_name = df_nodes[df_nodes['id'] == path[i]]['name'].values[0]
        mode = "Bus/Métro" if d['type'] == 'TRAVEL' else "Changement"
        arr_est = d['dep'] + d['weight'] # Votre logique de coût
        print(f"📍 {s_name[:34]:<35} | {mode:<12} | {str(d.get('line','---')):<10} | {fmt(arr_est)}")
    print(f"{'='*100}")
else:
    print("❌ Aucun itinéraire trouvé. Vérifiez que la ville couvre bien ces zones à 08h00.")

con.close()

🚀 Recherche optimisée pour Barcelone à 08:00:00


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Données chargées : 311383 trajets | 36568308 transferts légers
🌍 Localisation des quartiers...
⚙️ Analyse des combinaisons de stations...
❌ Aucun itinéraire trouvé. Vérifiez que la ville couvre bien ces zones à 08h00.


In [ ]:
import duckdb
import networkx as nx
import pandas as pd
import time

# --- 1. CONFIGURATION ---
PATH_BASE = "/content/drive/MyDrive/GTFS_FINAL/GRAPH_FINAL_SECURE_V3"
VILLE = "Catalonia Area de Barcelona"
VILLE_FOLDER = "Catalonia_Area_de_Barcelona"
t_sec = 36000 # 10h00
S_ID = "bgs_3421"

con = duckdb.connect()

# --- 2. TROUVER UNE DESTINATION GARANTIE (Même ligne) ---
print(f"🔍 Recherche d'un trajet passant par {S_ID}...")

# On cherche le premier trip_id qui passe par notre station
valid_trip = con.execute(f"""
    SELECT trip_id, route_id FROM read_parquet('{PATH_BASE}/EDGES_TRAVEL/*.parquet')
    WHERE source = '{S_ID}' AND dep_sec >= {t_sec}
    LIMIT 1
""").df()

if valid_trip.empty:
    print("❌ Aucun bus ne part de cette station à cette heure.")
else:
    MY_TRIP = valid_trip['trip_id'].iloc[0]
    print(f"✅ Trip trouvé : {MY_TRIP}")

    # On cherche toutes les stations desservies par ce trip APRÈS notre station
    df_targets = con.execute(f"""
        SELECT target FROM read_parquet('{PATH_BASE}/EDGES_TRAVEL/*.parquet')
        WHERE trip_id = '{MY_TRIP}' AND dep_sec >= {t_sec}
    """).df()

    T_ID = df_targets['target'].iloc[-1] # On prend la dernière station du bus
    print(f"🎯 Destination cible identifiée : {T_ID}")

    # --- 3. CHARGEMENT DU GRAPHE POUR CE TRAJET ---
    df_travel = con.execute(f"""
        SELECT CAST(source AS VARCHAR) as source, CAST(target AS VARCHAR) as target,
               weight_final, route_id, trip_id, dep_sec, arr_sec
        FROM read_parquet('{PATH_BASE}/EDGES_TRAVEL/*.parquet')
        WHERE city = '{VILLE}' AND dep_sec BETWEEN {t_sec} AND {t_sec + 7200}
    """).df()

    # On charge les noms des stations
    df_nodes = con.execute(f"SELECT id, name FROM read_parquet('{PATH_BASE}/NODES/*.parquet') WHERE city = '{VILLE}'").df()
    node_names = dict(zip(df_nodes.id, df_nodes.name))

    G = nx.DiGraph()
    for r in df_travel.itertuples():
        G.add_edge(r.source, r.target, weight=r.weight_final, trip=r.trip_id, dep=r.dep_sec)

    # --- 4. CALCUL DIJKSTRA ---
    try:
        path = nx.shortest_path(G, source=S_ID, target=T_ID, weight='weight')

        print(f"\n✅ ITINÉRAIRE TROUVÉ (Même ligne)")
        print(f"{'='*110}")
        print(f"{'STATION':<35} | {'DÉPART':<12} | {'ARRIVÉE RÉELLE (COÛT)'}")
        print(f"{'-'*110}")

        for i in range(len(path) - 1):
            data = G.get_edge_data(path[i], path[i+1])
            s_name = node_names.get(path[i], path[i])

            heure_dep = time.strftime('%H:%M:%S', time.gmtime(data['dep']))
            # Votre logique : Arrivée = Départ + Poids
            heure_arr = time.strftime('%H:%M:%S', time.gmtime(data['dep'] + data['weight']))

            print(f"📍 {s_name[:34]:<35} | {heure_dep:<12} | {heure_arr}")

        print(f"{'-'*110}")
        print(f"🏁 Arrivée finale à destination : {node_names.get(path[-1], path[-1])}")

    except Exception as e:
        print(f"❌ Erreur inattendue : {e}")

con.close()

🔍 Recherche d'un trajet passant par bgs_3421...
✅ Trip trouvé : bgs_07141041940
🎯 Destination cible identifiée : bgs_4897

✅ ITINÉRAIRE TROUVÉ (Même ligne)
STATION                             | DÉPART       | ARRIVÉE RÉELLE (COÛT)
--------------------------------------------------------------------------------------------------------------
📍 Carrer Barquera                     | 11:10:00     | 11:10:01
📍 Carrer de Sant Cristòfol            | 11:00:00     | 11:00:01
📍 Carrer del Progrés                  | 11:41:00     | 11:41:01
❌ Erreur inattendue : 'NoneType' object is not subscriptable


In [ ]:
import networkx as nx
import duckdb

# --- 1. CHARGEMENT ET CONSTRUCTION DU GRAPHE (Fenêtre large) ---
con = duckdb.connect()
PATH_BASE = "/content/drive/MyDrive/GTFS_FINAL/GRAPH_FINAL_SECURE_V3"
VILLE = "Catalonia Area de Barcelona"
t_sec = 36000 # 10h00

# On charge plus de données (4h au lieu de 2h) et plus de transferts (10 au lieu de 5)
df_travel = con.execute(f"SELECT source, target, weight_final FROM read_parquet('{PATH_BASE}/EDGES_TRAVEL/*.parquet') WHERE city = '{VILLE}' AND dep_sec BETWEEN {t_sec} AND {t_sec + 14400}").df()
df_transfer = con.execute(f"SELECT source, target, weight_final FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY source ORDER BY weight_final ASC) as rank FROM read_parquet('{PATH_BASE}/EDGES_TRANSFER/city=*.parquet')) WHERE rank <= 10").df()

G = nx.DiGraph()
for r in df_travel.itertuples(): G.add_edge(r.source, r.target, weight=r.weight_final)
for r in df_transfer.itertuples(): G.add_edge(r.source, r.target, weight=r.weight_final)

# --- 2. TEST DE CONNECTIVITÉ ---
S_ID = "bgs_3421"
T_ID = "reg_73001"

print(f"--- Diagnostic de connectivité ---")
if S_ID not in G:
    print(f"❌ La station de départ {S_ID} n'est même pas dans le graphe (aucun bus n'en part).")
elif T_ID not in G:
    print(f"❌ La station d'arrivée {T_ID} n'est pas dans le graphe (aucun bus n'y arrive).")
else:
    # On regarde toutes les stations accessibles depuis S_ID
    reachable_nodes = nx.descendants(G, S_ID)
    print(f"✅ Nombre de stations atteignables depuis {S_ID} : {len(reachable_nodes)}")

    if T_ID in reachable_nodes:
        print(f"🚀 Bonne nouvelle : {T_ID} EST atteignable ! Dijkstra devrait fonctionner.")
    else:
        print(f"❌ {T_ID} n'est physiquement pas reliée à {S_ID} dans ce créneau horaire.")
        # On affiche 5 stations qui sont atteignables pour vous donner des idées de test
        print(f"Exemples de stations atteignables : {list(reachable_nodes)[:5]}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

IOException: IO Error: No files found that match the pattern "/content/drive/MyDrive/GTFS_FINAL/GRAPH_FINAL_SECURE_V3/EDGES_TRANSFER/city=*.parquet"

In [ ]:
!ls "/content/drive/MyDrive/GTFS_FINAL/GRAPH_FINAL_SECURE_V3"

EDGES_TRANSFER	EDGES_TRAVEL  NODES


In [ ]:
import duckdb
import networkx as nx
import time
import gc
from geopy.geocoders import Nominatim
from scipy.spatial import KDTree

# --- 1. CONFIGURATION ---
PATH_BASE = "/content/drive/MyDrive/GTFS_FINAL/GRAPH_FINAL_SECURE_V3"
VILLE = "Catalonia Area de Barcelona"
VILLE_FOLDER = VILLE.replace(" ", "_").replace("(", "").replace(")", "")
HEURE_DEPART = "10:00:00"

h, m, s = map(int, HEURE_DEPART.split(':'))
t_sec = h * 3600 + m * 60 + s

con = duckdb.connect()

print(f"🚀 Démarrage sécurisé pour Barcelone...")

# --- 2. CHARGEMENT NODES ---
df_nodes = con.execute(f"SELECT id, name, lat, lon FROM read_parquet('{PATH_BASE}/NODES/*.parquet') WHERE city = '{VILLE}'").df()

# --- 3. RECHERCHE SPATIALE ---
spatial_tree = KDTree(df_nodes[['lat', 'lon']].values)
geolocator = Nominatim(user_agent="bcn_safe_v1", timeout=20)

def get_node(q):
    loc = geolocator.geocode(f"{q}, Barcelona, Spain")
    if not loc: return None
    return df_nodes.iloc[spatial_tree.query([loc.latitude, loc.longitude])[1]]

node_start = get_node("Plaça d'Espanya")
node_end = get_node("Sants Estació")

# --- 4. CONSTRUCTION DU GRAPHE (FLUX OPTIMISÉ) ---
G = nx.DiGraph()

print("🚌 Chargement des trajets (Travel)...")
# On limite à 1 heure de trajet
travel_cursor = con.execute(f"""
    SELECT source, target, weight_final, route_type, route_id, dep_sec, arr_sec
    FROM read_parquet('{PATH_BASE}/EDGES_TRAVEL/*.parquet')
    WHERE city = '{VILLE}' AND dep_sec BETWEEN {t_sec} AND {t_sec + 3600}
""")

for r in travel_cursor.fetchall():
    G.add_edge(r[0], r[1], weight=r[2], mode=r[3], line=r[4], dep=r[5], arr=r[6], type='TRAVEL')

print("🔄 Chargement limité des transferts (Top 200k)...")
# On ne prend que les transferts les plus courts pour économiser la RAM
transfer_cursor = con.execute(f"""
    SELECT source, target, weight_final, dep_sec, arr_sec
    FROM read_parquet('{PATH_BASE}/EDGES_TRANSFER/city={VILLE_FOLDER}/*.parquet')
    WHERE dep_sec BETWEEN {t_sec} AND {t_sec + 3600}
    ORDER BY weight_final ASC
    LIMIT 200000
""")

for r in transfer_cursor.fetchall():
    G.add_edge(r[0], r[1], weight=r[2], mode='TRANSFER', dep=r[3], arr=r[4], type='TRANSFER')

# Nettoyage agressif de la mémoire
del travel_cursor, transfer_cursor
gc.collect()

print(f"✅ Graphe prêt : {G.number_of_edges()} arêtes en mémoire.")

# --- 5. CALCUL ET AFFICHAGE ---
try:
    path = nx.shortest_path(G, node_start['id'], node_end['id'], weight='weight')

    print(f"\n✅ ITINÉRAIRE TROUVÉ")
    print("-" * 90)
    for i in range(len(path)-1):
        d = G.get_edge_data(path[i], path[i+1])
        s_name = df_nodes[df_nodes['id'] == path[i]]['name'].values[0]
        mode = "Bus/Métro" if d['type'] == 'TRAVEL' else "Changement"
        print(f"📍 {s_name[:30]:<30} | {mode:<12} | {time.strftime('%H:%M:%S', time.gmtime(d['dep']))}")

    final_arr = G.get_edge_data(path[-2], path[-1])['arr']
    print("-" * 90)
    print(f"🏁 Arrivée estimée : {time.strftime('%H:%M:%S', time.gmtime(final_arr))}")

except Exception as e:
    print(f"❌ Aucun trajet trouvé : {e}")

con.close()

🚀 Démarrage sécurisé pour Barcelone...
🚌 Chargement des trajets (Travel)...
🔄 Chargement limité des transferts (Top 200k)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Graphe prêt : 2656 arêtes en mémoire.
❌ Aucun trajet trouvé : Source ssi_8237 is not in G


In [ ]:
FIN

In [ ]:
# --- CONFIGURATION DES CHEMINS ---
INPUT_PATH = "/content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES_COSTS_CLEAN/*.parquet"
OUTPUT_BASE = "/content/drive/MyDrive/GTFS_FINAL/GRAPH_FINAL_SECURE"

# Création manuelle des dossiers pour être sûr qu'ils existent
os.makedirs(f"{OUTPUT_BASE}/NODES", exist_ok=True)
os.makedirs(f"{OUTPUT_BASE}/EDGES_TRAVEL", exist_ok=True)
os.makedirs(f"{OUTPUT_BASE}/EDGES_TRANSFER", exist_ok=True)

# --- FONCTION DE CONVERSION ---
def time_to_seconds(col_name):
    parts = F.split(F.col(col_name), ":")
    return (parts[0].cast("int") * 3600 + parts[1].cast("int") * 60 + parts[2].cast("int"))

# --- 2. CHARGEMENT ET CALCULS DES POIDS ---
print("Reading cleaned costs...")
df = spark.read.parquet(INPUT_PATH)

df = df.withColumn("dep_sec", time_to_seconds("from_departure_time")) \
       .withColumn("arr_sec", time_to_seconds("to_arrival_time"))

# Peak 1: 07h-09h | Peak 2: 17h-19h
df = df.withColumn("is_peak",
    ((F.col("dep_sec") >= 25200) & (F.col("dep_sec") <= 32400)) |
    ((F.col("dep_sec") >= 61200) & (F.col("dep_sec") <= 68400))
)

# Poids dynamique (Bus x3, Tram x1.2)
df = df.withColumn("weight_dynamic",
    F.when(F.col("is_peak"),
        F.when(F.col("route_type") == "3", F.col("cost_total") * 3)
         .when(F.col("route_type") == "0", F.col("cost_total") * 1.2)
         .otherwise(F.col("cost_total"))
    ).otherwise(F.col("cost_total"))
)

# --- 3. SAUVEGARDE DES NODES ET TRAVEL (Etape stable) ---
print("Saving Nodes and Travel Edges...")

# NODES (Stations)
nodes = df.select(F.col("from_stop_id").alias("id"), "name", "city",
                  F.col("from_lat").alias("lat"), F.col("from_lon").alias("lon")).distinct()
nodes.write.mode("overwrite").parquet(f"{OUTPUT_BASE}/NODES")

# TRAVEL EDGES (Le mouvement des bus)
edges_travel = df.select(
    F.col("from_stop_id").alias("source"),
    F.col("to_stop_id").alias("target"),
    "trip_id", "route_id", "route_type", "city",
    "dep_sec", "arr_sec", "weight_dynamic"
).withColumn("edge_type", F.lit("TRAVEL"))

edges_travel.write.mode("overwrite").parquet(f"{OUTPUT_BASE}/EDGES_TRAVEL")

# On libère la mémoire
del df, nodes
gc.collect()

Reading cleaned costs...
Saving Nodes and Travel Edges...


131

In [ ]:




# --- 4. GÉNÉRATION DES TRANSFERTS (BOUCLE VILLE PAR VILLE) ---
print("Starting safe city-by-city transfer generation...")

# Recharger les arêtes de trajet pour isoler la mémoire
travel_data = spark.read.parquet(f"{OUTPUT_BASE}/EDGES_TRAVEL")
cities = [row['city'] for row in travel_data.select("city").distinct().collect() if row['city'] is not None]

for i, city in enumerate(cities):
    # Flag de reprise
    city_safe_name = city.replace(" ", "_").replace("(", "").replace(")", "")
    done_flag = f"{OUTPUT_BASE}/EDGES_TRANSFER/{city_safe_name}_DONE.txt"

    if os.path.exists(done_flag):
        continue

    print(f"🔄 [{i+1}/{len(cities)}] Traitement : {city}")

    try:
        city_df = travel_data.filter(F.col("city") == city).cache()

        arrivals = city_df.select(F.col("target").alias("stop_id"), F.col("trip_id").alias("tr_from"), F.col("arr_sec").alias("t_arr"))
        departures = city_df.select(F.col("source").alias("stop_id"), F.col("trip_id").alias("tr_to"), F.col("dep_sec").alias("t_dep"))

        # Jointure des correspondances (2 min min, 20 min max)
        transfers = arrivals.join(departures, "stop_id") \
            .filter((F.col("tr_from") != F.col("tr_to")) &
                    (F.col("t_dep") >= F.col("t_arr") + 120) &
                    (F.col("t_dep") <= F.col("t_arr") + 1200))

        # Poids = attente (en min) + 5 min de pénalité
        final_transfers = transfers.withColumn("weight_dynamic", ((F.col("t_dep") - F.col("t_arr")) / 60) + 5) \
            .select(
                F.col("stop_id").alias("source"),
                F.col("stop_id").alias("target"),
                F.col("tr_from").alias("trip_id_from"),
                F.col("tr_to").alias("trip_id_to"),
                F.lit(city).alias("city"),
                F.col("t_arr").alias("dep_sec"),
                F.col("t_dep").alias("arr_sec"),
                "weight_dynamic"
            ).withColumn("edge_type", F.lit("TRANSFER"))

        # Sauvegarde
        output_city_path = f"{OUTPUT_BASE}/EDGES_TRANSFER/city={city_safe_name}"
        final_transfers.coalesce(1).write.mode("overwrite").parquet(output_city_path)

        # Flag de succès
        with open(done_flag, "w") as f: f.write("ok")

        city_df.unpersist()
        del transfers, final_transfers
        gc.collect()

    except Exception as e:
        print(f"❌ Crash sur {city}: {e}")

print(f"✅ TERMINÉ ! Votre graphe est disponible dans {OUTPUT_BASE}")

Starting safe city-by-city transfer generation...
🔄 [1/70] Traitement : Empresa Municipal de Transportes de Madrid (EMT Madrid)
🔄 [2/70] Traitement : Autoridad de Transporte Metropolitano del Area de Barcelona (ATM) Buses and trains in Catalonia (full version)
🔄 [3/70] Traitement : Generalitat of Catalonia (Intercity bus)
🔄 [4/70] Traitement : Catalonia Area de Barcelona
🔄 [5/70] Traitement : Ouigo
🔄 [6/70] Traitement : La Veloz SA (Buses from the Belchite countryside area to Zaragoza (C12))
🔄 [7/70] Traitement : dBus (Donostiabus)
🔄 [8/70] Traitement : Empresa Municipal de Transportes de Valencia (EMT Valencia)
🔄 [9/70] Traitement : Junta de Extremadura (Bus du gouvernement régional d’Estrémadure)
🔄 [10/70] Traitement : La Coruña Tram Company SA
🔄 [11/70] Traitement : Bizkaibus
🔄 [12/70] Traitement : Direxis TGO (Transportes Generales de Olesa)
🔄 [13/70] Traitement : Alavabus
🔄 [14/70] Traitement : Autocorb Coaches
🔄 [15/70] Traitement : Consorcio Regional de Transportes de Madrid 

In [ ]:
path_nodes = "/content/drive/MyDrive/GTFS_FINAL/GRAPH_FINAL_SECURE/NODES"
count = spark.read.parquet(path_nodes).count()

print(f"📍 Nombre total de stations (nœuds) dans le graphe : {count}")

📍 Nombre total de stations (nœuds) dans le graphe : 276060
